# Marketing Mix Registry Walkthrough

This notebook walks through registering a marketing mix model configuration with the Model Zoo registry API and executing a run using synthetic data. Every step mirrors the command-line helper but keeps the process transparent and reproducible.

## Prerequisites

- The Docker Compose stack (`docker compose up --build`) should be running. This provides the FastAPI registry at `http://localhost:8000` and PostgreSQL storage.
- The repository root is mounted in this notebook session so we can load the bundled configuration (`configs/marketing_mix.json`).
- Python package `requests` is available. The following cell installs it if missing.

In [ ]:
#!pip install requests --quiet

## 1. Configure API Target

Set the base URL for the API and paths to the configuration file. Adjust these if your registry runs elsewhere.

In [ ]:
import json
from pathlib import Path
import requests

BASE_URL = "http://localhost:8000"
CONFIG_PATH = Path("configs/marketing_mix.json")
ENV = "dev"  # environment label stored with the configuration

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Expected config at {CONFIG_PATH}")

print(f"API base URL: {BASE_URL}")
print(f"Using configuration: {CONFIG_PATH}")
print(f"Environment tag: {ENV}")

## 2. Register the Marketing Mix Configuration

The configuration contains:
- Model metadata and binding (`model` section)
- Hyperparameter schema and engine values (`hyperparameters`)
- Metric definitions and bindings (`metrics`)
- Variable definitions and optional groupings (`variables`)
- Default evaluation plan (`evaluation`)
- Execution context hints (`context`)

We POST this JSON to `/registry/configs`. The API stores it in Postgres, registers the model/metrics, and keeps the bindings in memory.

In [ ]:
config_payload = json.loads(CONFIG_PATH.read_text())
response = requests.post(
    f"{BASE_URL.rstrip('/')}/registry/configs",
    json={"env": ENV, "config": config_payload},
    timeout=30,
)
response.raise_for_status()
registered = response.json()
registered

## 3. Inspect the Stored Specification

Fetch the persisted model spec and associated configurations using `/registry/models/{model_id}`. This confirms the registry recorded hyperparameters, tags, and metric bindings.

In [ ]:
model_id = registered.get("model_id")
if not model_id:
    raise ValueError("Registry response missing model_id")

spec_response = requests.get(f"{BASE_URL.rstrip('/')}/registry/models/{model_id}", timeout=30)
spec_response.raise_for_status()
model_spec = spec_response.json()
model_spec

## 4. Generate Synthetic Marketing Mix Data

Recreate the helper from `bhm_pymc/marketing_mix_pymc_example.py`. The generator produces weekly channel spend for TV/Radio/Social and corresponding sales values. The data sources must match the `variables.definitions` in the configuration (e.g., `tv_spend`, `marketing_rows`, `sales`).

In [ ]:
from typing import List, Dict, Tuple
import random


def make_synthetic_marketing_mix_weeks(n_weeks: int = 52, seed: int = 42) -> Tuple[List[Dict[str, float]], List[float]]:
    random.seed(seed)
    tv = [random.uniform(5.0, 30.0) for _ in range(n_weeks)]
    radio = [random.uniform(1.0, 10.0) for _ in range(n_weeks)]
    social = [random.uniform(0.5, 8.0) for _ in range(n_weeks)]

    alpha = 100.0
    b_tv, b_radio, b_social = 2.0, 1.0, 0.5
    noise_sigma = 5.0

    sales = [alpha + b_tv * tv[i] + b_radio * radio[i] + b_social * social[i] + random.gauss(0, noise_sigma) for i in range(n_weeks)]
    rows = [{"tv": tv[i], "radio": radio[i], "social": social[i]} for i in range(n_weeks)]
    return rows, sales

rows, sales = make_synthetic_marketing_mix_weeks()
len(rows), rows[0], sales[0]

## 5. Submit a Run via `/runs`

Construct the payload expected by the new run endpoint:
- `model_id` and `env` select the stored configuration.
- `data_sources` provides the raw values referenced in the config (marketing rows, channel spends, and sales targets).
- `evaluation` overrides metric IDs if desired (here we re-use the config defaults).

The endpoint executes the model in-process using the registered binding, computes the metrics, and returns the summary.

In [ ]:
from itertools import repeat

# Build data sources to match the variable definitions (see configs/marketing_mix.json)
tv_spend = [row["tv"] for row in rows]
radio_spend = [row["radio"] for row in rows]
social_spend = [row["social"] for row in rows]

run_payload = {
    "model_id": model_id,
    "env": ENV,
    "data_sources": {
        "marketing_rows": rows,
        "sales": sales,
        "tv_spend": tv_spend,
        "radio_spend": radio_spend,
        "social_spend": social_spend,
    },
    "evaluation": {
        "metric_ids": ["r2_v1", "mse_v1"],
        "y_true_key": "y",
    },
}

run_response = requests.post(f"{BASE_URL.rstrip('/')}/runs", json=run_payload, timeout=600)
run_response.raise_for_status()
run_result = run_response.json()
run_result

## 6. Review Evaluation Metrics and Summary

The response includes:
- `status`: `succeeded`, `failed`, or `cancelled`
- `diagnostics`: coefficients and intercept emitted by the runner
- `eval_train` / `eval_valid`: metric scores on the splits
- `summary_text`: R-style summary (also available in `summary` as a dict)

Use this cell to display key pieces in a friendly format.

In [ ]:
from pprint import pprint

print(f"Run status: {run_result['status']}")
print("
Train metrics:")
pprint(run_result.get("eval_train", {}))
print("
Validation metrics:")
pprint(run_result.get("eval_valid", {}))

print("
Summary text:")
print(run_result.get("summary_text"))

## Next Steps

- Experiment with alternate configurations (change priors, transforms, or metrics) and re-run the notebook.
- Swap the synthetic generator for real weekly marketing data by populating the `data_sources` dict accordingly.
- Extend the API client with authentication or asynchronous run submission if you need production-grade orchestration.